In [ ]:
#python labelimg.py ./test_photo_rotated ./trainimg/classes.txt
#列出所有python環境
#conda info --envs

In [1]:
!pip install "numpy<2.0" scipy --force-reinstall

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
   ---------------------------------------- 0.0/38.6 MB ? eta -:--:--
   ---- ----------------------------------- 3.9/38.6 MB 29.4 MB/s eta 0:00:02
   ---------- ----------------------------- 9.7/38.6 MB 27.4 MB/s eta 0:00:02
   ------------------- -------------------- 18.4/38.6 MB 34.1 MB/s eta 0:00:01
   -------------------------- ------------- 25.2/38.6 MB 33.2 MB/s eta 0:00:01
   --------------------------------- ------ 32.8/38.6 MB 33.5 MB/s eta 0:00:01
   ---------------------------------------- 38.6/38.6 MB 33.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: scipy
    Found existing installation: scipy 1.13.1
    Uninstalling scipy-1.13.1:
      Successfully uninstalled scipy-1.13.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.3 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [8]:
import os
import shutil
import glob

# ================= 設定區域 =================
# 1. 來源與目的資料夾
SOURCE_FOLDER = "./labelImg-master/trainimg"      # 原始資料夾
DEST_FOLDER = "./labelImg-master/test_photo"      # 目的資料夾

# 2. 你要篩選的「類別 ID」 (請查看 classes.txt 確認對應數字)
# 範例：如果 level_1 是第 1 個 (ID 0)，level_2 是第 2 個 (ID 1)
# 這裡就填入 [0, 1]
TARGET_CLASS_IDS = [0, 1]  

# ===========================================

def copy_files_by_class():
    # 建立目的資料夾
    if not os.path.exists(DEST_FOLDER):
        os.makedirs(DEST_FOLDER)
        print(f"已建立資料夾: {DEST_FOLDER}")

    # 取得所有 txt 檔案
    txt_files = glob.glob(os.path.join(SOURCE_FOLDER, "*.txt"))
    print(f"正在掃描 {len(txt_files)} 個標註檔...")

    count = 0
    copied_count = 0

    for txt_path in txt_files:
        filename = os.path.basename(txt_path)
        base_name = os.path.splitext(filename)[0]
        
        should_copy = False
        
        # 1. 讀取 txt 檢查是否包含目標類別
        try:
            with open(txt_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    data = line.strip().split()
                    if len(data) > 0:
                        class_id = int(data[0]) # 讀取每一行的第一個數字
                        
                        if class_id in TARGET_CLASS_IDS:
                            should_copy = True
                            break # 只要找到其中一個符合的類別，就決定複製，跳出迴圈
        except Exception as e:
            print(f"讀取 {filename} 失敗: {e}")
            continue

        # 2. 如果符合條件，開始複製檔案
        if should_copy:
            # 複製 txt
            dest_txt_path = os.path.join(DEST_FOLDER, filename)
            shutil.copy(txt_path, dest_txt_path)
            
            # 尋找並複製對應的圖片 (支援多種副檔名)
            image_found = False
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
                img_name = base_name + ext
                src_img_path = os.path.join(SOURCE_FOLDER, img_name)
                
                if os.path.exists(src_img_path):
                    dest_img_path = os.path.join(DEST_FOLDER, img_name)
                    shutil.copy(src_img_path, dest_img_path)
                    image_found = True
                    break # 找到對應圖片就停止找副檔名
            
            if image_found:
                copied_count += 1
                # print(f"已複製: {base_name}") # 如果不想看太多訊息可以註解掉這行
            else:
                print(f"警告: 找到標註 {filename} 但找不到對應圖片！")

        count += 1
        if count % 100 == 0:
            print(f"已掃描 {count} 檔...")

    print("--------------------------------")
    print(f"處理完成！")
    print(f"總共掃描: {len(txt_files)} 檔")
    print(f"成功複製: {copied_count} 組 (圖片+txt)")
    print(f"檔案位於: {DEST_FOLDER}")

if __name__ == "__main__":
    copy_files_by_class()

正在掃描 484 個標註檔...
已掃描 100 檔...
已掃描 200 檔...
已掃描 300 檔...
已掃描 400 檔...
讀取 classes.txt 失敗: invalid literal for int() with base 10: 'level_1'
--------------------------------
處理完成！
總共掃描: 484 檔
成功複製: 367 組 (圖片+txt)
檔案位於: ./labelImg-master/test_photo


In [9]:
import os
import glob

# ================= 設定區域 =================
TARGET_FOLDER = "./labelImg-master/test_photo"   # 目標資料夾
TARGET_ID = 2                  # 要刪除的類別 ID
classes_file_name = "classes.txt" # 絕對不能刪除的檔案

# ★★★ 安全開關 ★★★
# True  = 真槍實彈，會直接刪除檔案
# False = 模擬測試，只會印出「準備刪除...」讓你檢查
PERFORM_DELETE = True  
# ===========================================

def delete_files_with_target_id():
    # 檢查資料夾是否存在
    if not os.path.exists(TARGET_FOLDER):
        print(f"找不到資料夾: {TARGET_FOLDER}")
        return

    # 取得所有 txt 檔
    txt_files = glob.glob(os.path.join(TARGET_FOLDER, "*.txt"))
    print(f"正在掃描 {len(txt_files)} 個檔案...\n")

    deleted_count = 0
    
    for txt_path in txt_files:
        # 1. 嚴格排除 classes.txt
        if os.path.basename(txt_path) == classes_file_name:
            continue

        base_name = os.path.splitext(os.path.basename(txt_path))[0]
        has_target_id = False

        # 2. 讀取內容檢查是否有目標 ID
        try:
            with open(txt_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    data = line.strip().split()
                    if len(data) > 0:
                        # 檢查每一行的第一個數字 (Class ID)
                        if int(data[0]) == TARGET_ID:
                            has_target_id = True
                            break
        except Exception as e:
            print(f"讀取錯誤 {txt_path}: {e}")
            continue

        # 3. 如果包含目標 ID，執行刪除 (包含圖片)
        if has_target_id:
            # 尋找對應的圖片
            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
                temp_path = os.path.join(TARGET_FOLDER, base_name + ext)
                if os.path.exists(temp_path):
                    img_path = temp_path
                    break
            
            # 執行動作
            if PERFORM_DELETE:
                # 刪除 txt
                os.remove(txt_path)
                print(f"[已刪除] 標註: {os.path.basename(txt_path)}")
                
                # 刪除圖片 (如果有的話)
                if img_path:
                    os.remove(img_path)
                    print(f"[已刪除] 圖片: {os.path.basename(img_path)}")
                else:
                    print(f"[提示] 圖片不存在，僅刪除 txt: {base_name}")
            else:
                # 模擬模式
                print(f"[模擬刪除] 發現 ID {TARGET_ID} -> {os.path.basename(txt_path)}")
                if img_path:
                    print(f"           連帶刪除 -> {os.path.basename(img_path)}")

            deleted_count += 1

    print("\n--------------------------------")
    if PERFORM_DELETE:
        print(f"處理完成！共刪除 {deleted_count} 組檔案。")
    else:
        print(f"模擬完成！若執行將會刪除 {deleted_count} 組檔案。")
        print("請將變數 PERFORM_DELETE 改為 True 來執行刪除。")

if __name__ == "__main__":
    delete_files_with_target_id()

正在掃描 368 個檔案...

[已刪除] 標註: 100.txt
[已刪除] 圖片: 100.jpg
[已刪除] 標註: 106.txt
[已刪除] 圖片: 106.jpg
[已刪除] 標註: 108.txt
[已刪除] 圖片: 108.jpg
[已刪除] 標註: 109.txt
[已刪除] 圖片: 109.jpg
[已刪除] 標註: 11.txt
[已刪除] 圖片: 11.jpg
[已刪除] 標註: 111.txt
[已刪除] 圖片: 111.jpg
[已刪除] 標註: 112.txt
[已刪除] 圖片: 112.jpg
[已刪除] 標註: 115.txt
[已刪除] 圖片: 115.jpg
[已刪除] 標註: 116.txt
[已刪除] 圖片: 116.jpg
[已刪除] 標註: 117.txt
[已刪除] 圖片: 117.jpg
[已刪除] 標註: 118.txt
[已刪除] 圖片: 118.jpg
[已刪除] 標註: 119.txt
[已刪除] 圖片: 119.jpg
[已刪除] 標註: 12.txt
[已刪除] 圖片: 12.jpg
[已刪除] 標註: 120.txt
[已刪除] 圖片: 120.jpg
[已刪除] 標註: 122.txt
[已刪除] 圖片: 122.jpg
[已刪除] 標註: 125.txt
[已刪除] 圖片: 125.jpg
[已刪除] 標註: 126.txt
[已刪除] 圖片: 126.jpg
[已刪除] 標註: 130.txt
[已刪除] 圖片: 130.jpg
[已刪除] 標註: 131.txt
[已刪除] 圖片: 131.jpg
[已刪除] 標註: 14.txt
[已刪除] 圖片: 14.jpg
[已刪除] 標註: 145.txt
[已刪除] 圖片: 145.jpg
[已刪除] 標註: 146.txt
[已刪除] 圖片: 146.jpg
[已刪除] 標註: 148.txt
[已刪除] 圖片: 148.jpg
[已刪除] 標註: 169.txt
[已刪除] 圖片: 169.jpg
[已刪除] 標註: 170.txt
[已刪除] 圖片: 170.jpg
[已刪除] 標註: 171.txt
[已刪除] 圖片: 171.jpg
[已刪除] 標註: 172.txt
[已刪除] 圖片: 172.jpg
[已刪除] 標註: 176.txt

In [14]:
import albumentations as A
import cv2
import os
import glob
import numpy as np

# ================= 設定區域 (修改這裡就好) =================
# 1. 輸入與輸出資料夾
INPUT_FOLDER = "labelImg-master\\test_photo"        # 你的原始資料夾
OUTPUT_FOLDER = "labelImg-master\\test_photo_rotated" # 輸出的資料夾
ROTATION_LIMIT = 60   # 旋轉角度
BORDER_VALUE = (114, 114, 114) # 填充背景顏色
# =========================================================

def process_batch():
    # 建立輸出資料夾
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
        print(f"已建立輸出資料夾: {OUTPUT_FOLDER}")

    # ================= 修正後的增強管線 =================
    # 1. 使用 A.Compose 包裹
    # 2. 設定 bbox_params 以支援 YOLO 格式和 class_labels
    transform = A.Compose([
        A.ShiftScaleRotate(
            rotate_limit=ROTATION_LIMIT,
            scale_limit=0,      # Disable scaling
            shift_limit=0,      # Disable shifting
            p=1.0,
            border_mode=cv2.BORDER_CONSTANT,
            value=BORDER_VALUE
        )
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
    # ======================================================

    # 取得所有圖片
    image_paths = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG']:
        image_paths.extend(glob.glob(os.path.join(INPUT_FOLDER, ext)))

    print(f"找到 {len(image_paths)} 張圖片，準備開始處理...\n")

    count = 0
    for img_path in image_paths:
        try:
            # 1. 讀取圖片
            image = cv2.imread(img_path)
            if image is None:
                continue
            
            # 2. 讀取對應的 txt 標註檔
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            txt_path = os.path.join(INPUT_FOLDER, base_name + ".txt")
            
            bboxes = []
            class_labels = []
            
            if os.path.exists(txt_path):
                with open(txt_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        data = line.strip().split()
                        if len(data) >= 5:
                            cls = int(data[0])
                            # YOLO 格式: class x_center y_center width height
                            x, y, w, h = map(float, data[1:5])
                            bboxes.append([x, y, w, h])
                            class_labels.append(cls)
            
            # 3. 執行旋轉
            # Albumentations 在 bbox 為空時仍可運作，但需傳入空列表
            transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            
            aug_image = transformed['image']
            aug_bboxes = transformed['bboxes']
            aug_labels = transformed['class_labels']

            # 4. 存檔
            out_img_name = base_name + "_rot.jpg"
            out_txt_name = base_name + "_rot.txt"
            
            # 儲存圖片
            cv2.imwrite(os.path.join(OUTPUT_FOLDER, out_img_name), aug_image)
            
            # 儲存標註
            if len(aug_bboxes) > 0:
                with open(os.path.join(OUTPUT_FOLDER, out_txt_name), 'w') as f:
                    for bbox, cls in zip(aug_bboxes, aug_labels):
                        x, y, w, h = bbox
                        
                        # 限制數值在 0-1 之間 (避免浮點數溢出導致 YOLO 訓練報錯)
                        x = min(max(x, 0.0), 1.0)
                        y = min(max(y, 0.0), 1.0)
                        w = min(max(w, 0.0), 1.0)
                        h = min(max(h, 0.0), 1.0)
                        
                        # 避免出現寬或高為 0 的無效框
                        if w > 0 and h > 0:
                            line = f"{int(cls)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n"
                            f.write(line)
            
            # 若原圖有 txt 但旋轉後沒框 (或原圖是空的 txt)，產生一個空檔案
            elif os.path.exists(txt_path):
                 open(os.path.join(OUTPUT_FOLDER, out_txt_name), 'w').close()

            count += 1
            if count % 10 == 0:
                print(f"已處理 {count} 張...")

        except Exception as e:
            print(f"處理 {img_path} 時發生錯誤: {e}")
            import traceback
            traceback.print_exc() # 印出詳細錯誤以便除錯

    print(f"\n完成！共處理 {count} 張圖片。")
    print(f"結果已儲存在: {OUTPUT_FOLDER}")

if __name__ == "__main__":
    process_batch()

C:\Users\0419mch\AppData\Local\Temp\ipykernel_17248\1558463623.py:25: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(


找到 516 張圖片，準備開始處理...

已處理 10 張...
已處理 20 張...
已處理 30 張...
已處理 40 張...
已處理 50 張...
已處理 60 張...
已處理 70 張...
已處理 80 張...
已處理 90 張...
已處理 100 張...
已處理 110 張...
已處理 120 張...
已處理 130 張...
已處理 140 張...
已處理 150 張...
已處理 160 張...
已處理 170 張...
已處理 180 張...
已處理 190 張...
已處理 200 張...
已處理 210 張...
已處理 220 張...
已處理 230 張...
已處理 240 張...
已處理 250 張...
已處理 260 張...
已處理 270 張...
已處理 280 張...
已處理 290 張...
已處理 300 張...
已處理 310 張...
已處理 320 張...
已處理 330 張...
已處理 340 張...
已處理 350 張...
已處理 360 張...
已處理 370 張...
已處理 380 張...
已處理 390 張...
已處理 400 張...
已處理 410 張...
已處理 420 張...
已處理 430 張...
已處理 440 張...
已處理 450 張...
已處理 460 張...
已處理 470 張...
已處理 480 張...
已處理 490 張...
已處理 500 張...
已處理 510 張...

完成！共處理 516 張圖片。
結果已儲存在: labelImg-master\test_photo_rotated


In [13]:
import os
import sys

# --- 腳本設定 ---
# 您的標註檔 (e.g., image001.txt) 所在的資料夾
DATA_DIR = './labelImg-master/test_photo_rotated' 

# 您的類別定義檔 (classes.txt) 的完整路徑
CLASSES_FILE = './labelImg-master/test_photo_rotated/classes.txt'

# 支援的圖片副檔名
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
# --- 結束設定 ---


def analyze_labels_final():
    """
    統計標註資料夾中的圖片總數、已標註數量、空標註數量與各類別的物件數量。
    """
    class_names = []
    
    # --- 步驟 1: 讀取 classes.txt ---
    try:
        with open(CLASSES_FILE, 'r', encoding='utf-8') as f:
            class_names = [line.strip() for line in f if line.strip()]
        
        if not class_names:
            print(f"錯誤: 類別檔案 '{CLASSES_FILE}' 是空的。")
            return
        print(f"成功讀取 {len(class_names)} 個類別: {class_names}")
    
    except FileNotFoundError:
        print(f"錯誤: 找不到類別檔案 '{CLASSES_FILE}'")
        return
    except Exception as e:
        print(f"讀取類別檔案時發生錯誤: {e}")
        return

    # --- 步驟 2: 初始化計數器 ---
    class_counts = {name: 0 for name in class_names}
    total_objects = 0
    
    # 集合初始化
    annotated_files_base = set()      # 有 .txt 檔案的圖片 (無論內容有無)
    empty_annotation_files = set()    # 有 .txt 但內容為空的圖片 (負樣本)
    total_image_files_base = set()    # 所有圖片檔案
    
    classes_filename = os.path.basename(CLASSES_FILE)

    # --- 步驟 3: 遍歷資料夾 ---
    try:
        all_files = os.listdir(DATA_DIR)
    except FileNotFoundError:
        print(f"錯誤: 找不到資料夾 '{DATA_DIR}'")
        return

    print(f"\n正在分析 '{DATA_DIR}' 中的檔案...")

    for filename in all_files:
        file_base_name, file_extension = os.path.splitext(filename)
        file_extension_lower = file_extension.lower()

        # 1. 收集所有圖片
        if file_extension_lower in IMAGE_EXTENSIONS:
            total_image_files_base.add(file_base_name)
            continue

        # 2. 檢查標註檔
        if file_extension_lower == '.txt' and filename != classes_filename:
            annotated_files_base.add(file_base_name)
            
            file_path = os.path.join(DATA_DIR, filename)
            
            try:
                with open(file_path, 'r') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    
                    # 檢查是否為空檔案
                    if not lines:
                        empty_annotation_files.add(file_base_name)
                        continue

                    # 統計類別
                    for line in lines:
                        parts = line.split()
                        try:
                            class_index = int(parts[0])
                            if 0 <= class_index < len(class_names):
                                class_name = class_names[class_index]
                                class_counts[class_name] += 1
                                total_objects += 1
                        except ValueError:
                            pass # 忽略格式錯誤
                            
            except Exception:
                pass # 忽略讀取錯誤

    # --- 步驟 4: 計算統計數據 ---
    
    # 總圖片數
    total_image_count = len(total_image_files_base)
    
    # 有對應 .txt 的圖片數 (包含空標註)
    has_txt_count = len(annotated_files_base.intersection(total_image_files_base))
    
    # 完全沒標註 (沒 .txt) 的圖片數
    missing_txt_count = total_image_count - has_txt_count
    
    # 空標註 (有 .txt 但沒框) 的數量
    empty_txt_count = len(empty_annotation_files)

    # --- 步驟 5: 輸出結果 ---
    print("\n" + "="*40)
    print("--- 標註狀態統計 ---")
    print("="*40)
    
    if total_image_count == 0:
        print("未找到任何圖片。")
    else:
        # 進度條
        progress_pct = (has_txt_count / total_image_count) * 100
        bar_len = 25
        filled = int(bar_len * has_txt_count // total_image_count)
        bar = '█' * filled + '-' * (bar_len - filled)
        
        print(f"總圖片數量: {total_image_count} 張")
        print(f"標註進度:   [{bar}] {progress_pct:.1f}%")
        print("-" * 40)
        print(f"✅ 已標註圖片 (有 .txt):      {has_txt_count} 張")
        print(f"❌ 未標註圖片 (無 .txt):      {missing_txt_count} 張")
        print(f"⚪ 空標註圖片 (無框/負樣本):  {empty_txt_count} 張") 

    print("\n" + "="*40)
    print(f"已標註物件總數 (Bounding Boxes): {total_objects}")
    print("-" * 40)
    print("--- 各類別物件數量 ---")
    
    if total_objects > 0:
        try:
            max_len = max(len(name) for name in class_names) + 2
        except:
            max_len = 10
            
        for name, count in class_counts.items():
            pct = (count / total_objects) * 100
            print(f"  - {name:<{max_len}} {count:<5} 個 ({pct:.1f}%)")
    else:
        print("沒有任何物件被標註。")

    print("="*40)
    print("統計完成。")

if __name__ == "__main__":
    analyze_labels_final()

成功讀取 3 個類別: ['level_1', 'level_2', 'level_3']

正在分析 './labelImg-master/test_photo_rotated' 中的檔案...

--- 標註狀態統計 ---
總圖片數量: 384 張
標註進度:   [█████████████████████████] 100.0%
----------------------------------------
✅ 已標註圖片 (有 .txt):      384 張
❌ 未標註圖片 (無 .txt):      0 張
⚪ 空標註圖片 (無框/負樣本):  2 張

已標註物件總數 (Bounding Boxes): 698
----------------------------------------
--- 各類別物件數量 ---
  - level_1   321   個 (46.0%)
  - level_2   377   個 (54.0%)
  - level_3   0     個 (0.0%)
統計完成。
